In [ ]:
pip install chembl_webresource_client pandas

In [ ]:
from chembl_webresource_client.new_client import new_client

# Search for your target — let's use malaria (Plasmodium falciparum)
target = new_client.target
results = target.search('Plasmodium falciparum')

# See what came back
for r in results[:5]:
    print(r['target_chembl_id'], r['pref_name'], r['target_type'])

In [ ]:
activity = new_client.activity

# Fetch IC50 data for your chosen target
data = activity.filter(
    target_chembl_id='CHEMBL364',  # replace with your chosen ID
    standard_type='IC50'
)

# Convert to a pandas dataframe
import pandas as pd
df = pd.DataFrame.from_records(data)

print(df.shape)
print(df.head())

In [ ]:
print(df.columns.tolist())

In [ ]:
# Keep only the columns you need
df = df[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'standard_units']]

# Drop rows with missing values
df = df.dropna(subset=['canonical_smiles', 'standard_value'])

# Convert IC50 to numeric
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df.dropna(subset=['standard_value'])

print(df.shape)
print(df.head())

In [ ]:
# Standard threshold: IC50 < 1000 nM = Active
df['label'] = df['standard_value'].apply(lambda x: 1 if x < 1000 else 0)

print(df['label'].value_counts())

In [ ]:
!pip install rdkit
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fingerprint = generator.GetFingerprint(mol)
    return list(fingerprint)

df['fingerprint'] = df['canonical_smiles'].apply(smiles_to_fingerprint)
df = df.dropna(subset=['fingerprint'])

print(df.shape)

In [ ]:
X = np.array(df['fingerprint'].tolist())  # features
y = df['label'].values                     # 0 or 1

print(X.shape)  # should be (n_compounds, 2048)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

!pip install imbalanced-learn

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Handle class imbalance
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(X_train.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, rf_preds))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

# XGBoost
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)

print("=== XGBoost ===")
print(classification_report(y_test, xgb_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, rf.predict_proba(X_test)[:,1])
plt.plot(fpr, tpr, label='Random Forest')

fpr2, tpr2, _ = roc_curve(y_test, xgb.predict_proba(X_test)[:,1])
plt.plot(fpr2, tpr2, label='XGBoost')

plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Drug Activity Prediction')
plt.legend()
plt.savefig('roc_curve.png')
plt.show()